# Lekcia 16 - Nasadenie škálovateľných agentov pomocou Microsoft Foundry

V tomto zošite vytvoríte **produkčne pripraveného agenta zákazníckej podpory** pre fiktívnu spoločnosť **Contoso**. Na rozdiel od predchádzajúcich lekcií, bodom tu nie je samotný agentov cyklus uvažovania — ide o všetko *okolo* neho, čo robí agenta bezpečným na spustenie vo veľkom:

1. **Volanie nástrojov** — vyhľadávanie objednávok a vytváranie tiketov.
2. **RAG** — odpovede na základe zásad z vedomostnej databázy.
3. **Pamäť** — zapamätanie si zákazníka naprieč kolami konverzácie.
4. **Smerovanie modelov** — posielanie jednoduchých požiadaviek na malý model, zložitých na veľký model.
5. **Cache odpovedí** — poskytovanie opakujúcich sa otázok bez volania modelu.
6. **Schválenie človekom** — vrátenia peňazí nad určitý limit čakajú na schválenie.
7. **Evaluačná brána** — offline testovacia sada, ktorá blokuje zlú verziu.
8. **Sledovateľnosť** — OpenTelemetry trasovanie okolo každej požiadavky.

Každá sekcia je samostatná a spustiteľná. Prečítajte si každý riadok — produkčné primitíva sú zámerne malé.


## Nastavenie

Pred spustením tohto notebooku sa uistite, že máte:

1. **Projekt Microsoft Foundry** s nasadeným chatovacím modelom (napr. `gpt-4.1-mini`).
2. **Prihlásenie prostredníctvom Azure CLI** — spustite `az login` vo svojom termináli.
3. **Nastavené požadované premenné prostredia:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint vášho projektu Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — názov vášho nasadeného modelu.

Sekcia RAG používa **Azure AI Search** v prípade, že sú nastavené `AZURE_SEARCH_SERVICE_ENDPOINT` a `AZURE_SEARCH_API_KEY`, inak používa vyhľadávanie v pamäti, aby notebook bežal aj bez vyhľadávacieho zdroja.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Nástroje

Produkčné nástroje vykonávajú skutočnú prácu so skutočnými systémami. Tu simulujeme databázu objednávok a systém na vystavovanie vstupeniek pomocou obyčajných Python funkcií. Dekorátor `@tool` ich sprístupňuje agentovi.

Všimnite si, že `issue_refund` používa `approval_mode="always_require"` pre refundácie nad určitý prah — toto je primitivum "človek v slučke", ktoré neskôr nasadzujeme.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Politická znalostná databáza

Politické otázky („aká je vaša lehota na vrátenie?“) by mali byť zodpovedané z autoritatívneho zdroja, nie z pamäte modelu. Obalíme malú znalostnú databázu ako vyhľadávací nástroj.

V produkcii je to **Azure AI Search**; tu poskytujeme vyhľadávanie kľúčových slov v pamäti, aby sa notebook spustil kdekoľvek, a automaticky prepneme na Azure AI Search, keď sú prítomné premenné prostredia.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Pamäť

Podporný agent, ktorý zabúda, s kým hovorí, je zlý podporný agent. Uchovávame malú databázu profilov na zákazníka a vkladáme krátky súhrn do inštrukcií agenta. V produkcii je to služba pamäti (pozri Lekcia 13); tu slovník robí vzor viditeľným.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Modelovanie smerovania a ukladanie odpovedí do vyrovnávacej pamäte

Dva nákladové prvky pripojené ku jednému spracovateľovi požiadaviek:

- **Smerovanie**: lacný heuristický klasifikátor rozhoduje, či požiadavka potrebuje malý alebo veľký model.
- **Ukladanie do vyrovnávacej pamäte**: normalizované opakujúce sa otázky sa podávajú priamo z vyrovnávacej pamäte bez volania modelu.

Klasifikátor je tu zámerne jednoduchý. Vo výrobe by ste ho validovali na základe prevádzky a mohli ho nahradiť Model Routerom Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agent, ľudské schválenie a pozorovateľnosť

Teraz zostavíme agenta z vyššie uvedených nástrojov a každý požiadavok zabalíme do OpenTelemetry spanu. Funkcia `handle_support_request` je produkčný spracovateľ požiadaviek: cache → route → trace → run → cache.

Ľudské schválenie je riešené rámcom: pretože `issue_refund` má `approval_mode="always_require"`, beh sa pozastaví a zobrazí požiadavku na schválenie, ktorú explicitne vyriešime.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Brána hodnotenia

Toto je brána vydania z lekcie: offline testovacia sada hodnotí agenta a nasadenie pokračuje iba vtedy, ak miera úspešnosti prekročí stanovený prah. Hodnotiteľom tu je jednoduchá kontrola prekrytia kľúčových slov, aby bol zošit samostatný; v produkcii by ste použili LLM ako rozhodcu alebo hodnotiaci framework (viď Lekcia 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Zostavenie dohromady: Simulované vydanie

Nasledujúca bunka ukazuje celý cyklus, ktorý lekcia popisuje: spustiť hodnotiace bránu a „nasadiť“ iba vtedy, ak prejde. Toto je vzor, ktorý by ste spustili v CI pred propagovaním verzie agenta do Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Zhrnutie

Zostavili ste zákaznícky podporný agent pripravený na produkciu so zapojením všetkých prevádzkových náležitostí:

- **Nástroje, RAG a pamäť** dávajú agentovi schopnosti a kontext.
- **Smerovanie modelov a ukladanie do vyrovnávacej pamäte** udržiavajú latenciu a náklady pod kontrolou.
- **Ľudské schválenie** stráži vysoko rizikové akcie, ako sú veľké vrátenia peňazí.
- **Vyhodnocovacia brána** blokuje zlé vydania pred ich spustením.
- **Sledovanie** robí každú požiadavku pozorovateľnou.

### Výzva

Rozšírte tohto agenta o:

1. **Podporu viacerých modelov** — pridajte tretí „rozumový“ stupeň a nasmerujte naň eskalácie/reklamácie.
2. **Pridajte vyhodnocovacie brány** — rozšírte `TEST_CASES` o scenáre schvaľovania vrátení peňazí a potvrďte, že brána zachytí regresie.
3. **Pridajte smerovanie s uvedomením si nákladov** — sledujte odhadované náklady na požiadavku (malé vs veľké vs vyrovnávacia pamäť) a vytlačte správu o nákladoch po sérii zmiešaných dotazov.

V ďalšej lekcii sa vyberiete opačnou cestou a spustíte agenta úplne na vlastnom počítači pomocou Microsoft Foundry Local a Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
